In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#change crest import to appropriate location
%cd -q ../..
from crest import HierarchalTensorGraph
from math import exp

## Simple HTG basenodes for addition and multiplication

In [ ]:
class Add(HierarchalTensorGraph):
    def __init__(self,name):
        self.name = name
        
        #call the basenode constructor
        super().__init__(lambda X : {'sum' :  X['x_1'] + X['x_2']},
                         name=name,
                         inputs={'x_1' : None, 'x_2' : None},
                         outputs={'sum' : None}
                        )
        
class Multiply(HierarchalTensorGraph):
    def __init__(self,name):
        self.name = name
        
        #call the basenode constructor
        super().__init__(lambda X : {'product' : X['scalar']*X['x']},
                         name=name,
                         inputs={'scalar' : None, 'x' : None},
                         outputs={'product' : None}
                        )

## A Coupled HTG that to perform a*(b+c)

In [ ]:
class AddMult(HierarchalTensorGraph):
    """Crest HTG to add and multiply a number"""
    
    def __init__(self, name):
        super().__init__(name=name,
                         inputs={'scalar' : None,'x_1' : None, 'x_2' : None},
                         outputs={'add_mult' : None}
                        )
        add = Add('adder')
        mult = Multiply('multiplier')
        self.add_edge('input',add)
        self.add_edge('input',mult)
        self.add_edge(add,mult)
        self.add_edge(mult,'output')
        
        #renaming of keys
        self.rename_io(inputs_map={'sum' : 'x'}, outputs_map={'product' : 'add_mult'},node=mult)

In [ ]:
#create instance
X = {'x_1' : 0.8, 'x_2' : 1.2, 'scalar' : 1.3}
add_mult = AddMult('add_mult')
add_mult(X)

## A HTG that combines for exp(a*(b+c))

In [ ]:
class Exp(HierarchalTensorGraph):
    def __init__(self,name):
        self.name = name
        
        #call the basenode constructor
        super().__init__(lambda X : {'exp' : exp(X['x'])},
                         name=name,
                         inputs={'x' : None},
                         outputs={'exp' : None}
                        )
class AddMultExp(HierarchalTensorGraph):
    
    def __init__(self):
        super().__init__(name='add_mult_exp',
                         inputs = {'x_1' : None, 'x_2' : None, 'scalar' : None},
                         outputs = {'add_mult_exp' : None}
                        )
        
        #define graph model
        self.add_edge('input',AddMult('add_mult'))
        self.add_edge('add_mult',Exp('exponentiate'))
        self.rename_io(inputs_map={'add_mult':'x'},outputs_map={'exp':'add_mult_exp'},node='exponentiate')
        self.add_edge('exponentiate','output')
        

In [ ]:
add_mult_exp = AddMultExp() 
add_mult_exp(X)

In [ ]:
#draw the graph
add_mult_exp.draw()

In [ ]:
#draw the graph with add_mult expanded
add_mult_exp.draw(expand_nodes='add_mult')

In [ ]:
#draw the graph with everything expanded
add_mult_exp.draw(expand_nodes='all')